# R04 · Cookbook-Referenzlabor: NumPy-Batch-API und Layouts

**NHyDB-0.27.1-Notebook-Reihe · v1 · Referenzteil**

Diese Beispiele basieren auf dem offiziellen
[NHyDB Python Cookbook](https://github.com/nhydb/NHyDB/blob/main/docs/cookbook/PYTHON_COOKBOOK.md) (Abschnitte 16–17), das als maßgebliche Referenz dient.
Code und erwartete Ausgaben sind unverändert aus dem Cookbook übernommen, die Begleittexte ins
Deutsche übersetzt; dieselben Beispiele führt auch die NHyDB-Testsuite aus.

* Jede Code-Zelle ist ein **eigenständiges Beispiel** – führe eine Zelle einzeln aus oder nutze *Run All*.
* Jedes Beispiel legt seinen eigenen temporären Datenbankordner an.
* Unter jedem Beispiel steht die im Cookbook abgedruckte Ausgabe, zum Vergleich mit deiner.
* Die Hauptreihe (`../main/`) erklärt die Ideen hinter diesen APIs Schritt für Schritt.

**Inhalt** (6 ausführbare Beispiele)

* 16. NumPy-Batch-API
* 17. Layouts: COLUMN, PACKED, CUSTOM, Routing und Weiterentwicklung

In [1]:
import nhydb

print('nhydb', nhydb.ENGINE_VERSION, '· file format', nhydb.FILE_FORMAT_VERSION)
if nhydb.ENGINE_VERSION != '0.27.1':
    print('⚠ These examples were written for NHyDB 0.27.1; results may differ.')

nhydb 0.27.1 · file format 0.16


## 16. NumPy-Batch-API

Die NumPy-Methoden kopieren ganze Spalten in die Engine; Kodierung und
Schreibschleifen laufen in Rust. Alle Beispiele in diesem Abschnitt brauchen NumPy.

In [2]:
# requires: numpy
import os
import tempfile

import numpy as np

import nhydb

path = os.path.join(tempfile.mkdtemp(), "ts.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "measurements")
    db.create_shape(1, 1, "sample", [
        {"v": 1, "name": "t", "type": "u64"},
        {"v": 2, "name": "temp", "type": "f32"},
        {"v": 3, "name": "ok", "type": "bool"},
        {"v": 4, "name": "station", "type": "utf8"},
    ])
    db.create_classic_layout(1, 1, "samples", shape_id=1)

    n = 100_000
    t = np.arange(n, dtype=np.uint64) * 60
    cols = {
        "t": t,
        "temp": (20 + np.sin(t / 3600.0)).astype(np.float32),
        "ok": t % 7 != 0,
        "station": np.array(["A", "B"] * (n // 2), dtype=object),
    }
    db.put_numpy_batch(1, 1, cols, commit=True)        # records u = 1 … 100000

    out = db.read_numpy(1, 99_999, 3, ["t", "temp", "station"])
    print(out["t"], out["temp"].dtype, list(out["station"]), out["_present"])

[5999880 5999940       0] float32 ['A', 'B', None] [ True  True False]


*Erwartete Ausgabe (aus dem Cookbook):*

```text
[5999880 5999940       0] float32 ['A', 'B', None] [ True  True False]
```

`read_numpy` liefert ein Array pro Feld plus `_present`; fehlende Datensätze
haben `_present=False` (Werte 0, NaN oder `None`). Ganzzahltypen kommen als
`int64` zurück (`varuint` als `uint64`), `utf8`/`bytes`/Enums als Object-Arrays.

Objekte beliebiger Layouts, dünn besetzte Zellen und Abfrageergebnisse:

In [3]:
# requires: numpy
import os
import tempfile

import numpy as np

import nhydb
from nhydb.query import Q, QueryEngine

path = os.path.join(tempfile.mkdtemp(), "np2.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "id", "type": "varuint"}, {"v": 2, "name": "x", "type": "f64"}])
    db.create_classic_layout(1, 1, "points", shape_id=1)

    ids = np.array([10, 11, 12, 50], dtype=np.uint64)
    db.put_objects_numpy(1, ids, {"id": ids, "x": ids / 4})                  # any layout, by object id
    db.put_cells_numpy(1, np.full(3, 500, dtype=np.uint64), np.array([1, 2, 3], dtype=np.uint64),
                       np.array([1.5, 2.5, 3.5]), type="f64", lane="NR")   # sparse cells
    db.commit()

    print(db.get_record(1, 50)["x"], db.get_value(1, 500, 3, lane="NR"))
    print(db.sql_numpy("SELECT id, x FROM points WHERE id < ? ORDER BY id", [50]))
    r = QueryEngine(db).execute_numpy(Q.field("x") > 2.8, realm=1, select=["id"])
    print(r["id"], QueryEngine(db).execute_sql_numpy("SELECT x FROM points WHERE id = 11")["x"])

12.5 3.5
{'id': array([10, 11, 12]), 'x': array([2.5 , 2.75, 3.  ])}
[12 50] [2.75]


*Erwartete Ausgabe (aus dem Cookbook):*

```text
12.5 3.5
{'id': array([10, 11, 12]), 'x': array([2.5 , 2.75, 3.  ])}
[12 50] [2.75]
```

Akzeptierte Spaltentypen: Ganzzahl-Arrays jeder Breite und Bytereihenfolge (Wertebereiche
werden geprüft), Float-Arrays, Bool-Arrays, Object-Arrays oder Listen für Text,
Bytes und Enum-Bezeichnungen. `u`/`v`/Objekt-ID-Arrays sind auf `uint64` begrenzt;
für größere Koordinaten nimm Listen von Python-`int`. Ungültige Eingaben lösen
`ValueError` (falsche Länge, außerhalb des Bereichs) oder `SchemaError` (fehlendes Feld) aus,
bevor irgendetwas geschrieben wird. `put_math_numpy` wird in Abschnitt 18 gezeigt.

## 17. Layouts: COLUMN, PACKED, CUSTOM, Routing und Weiterentwicklung

| Layout | Datensatz-ID | Feld | Verwenden für |
|---|---|---|---|
| CLASSIC | `u` | `v` (Shape-Feld) | gewöhnliche Tabellen (Abschnitte 3–8) |
| COLUMN | `v` | ein festes `u` pro Feld | spaltenorientierte Daten: alle Werte eines Felds teilen sich ein `u` |
| PACKED | `⌊(v−1)/span⌋ + 1` | Versatz in einem Block aus `span` Zellen | viele kleine Datensätze, entlang `v` eines `u` gepackt |
| CUSTOM | dein Plugin | dein Plugin | jede Abbildung, die du in Python implementierst |

`put_object`, `put_objects_batch`, `get_object` und `delete_record` funktionieren
für jedes Layout; `put_record`/`get_record` sind dieselben Aufrufe mit
einfacheren Namen.

In [4]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "layouts.nhydb")
with nhydb.create(path) as db:
    db.create_realm(2, "columns")
    db.create_column_layout(10, 2, "metrics", fields=[
        {"u": 1, "name": "temp", "type": "f64"},
        {"u": 2, "name": "humidity", "type": "u8"},
    ])
    db.put_object(10, 42, {"temp": 21.5, "humidity": 40})
    print(db.get_object(10, 42))
    print(db.get_object(10, 42, include_addresses=True)["_addresses"]["temp"]["u"])

    db.create_realm(3, "packed")
    db.create_packed_layout(11, 3, "pairs", u=1, span=8, fields=[
        {"offset": 0, "name": "a", "type": "varuint"},
        {"offset": 1, "name": "b", "type": "utf8"},
    ])
    print(db.put_objects_batch(11, [(1, {"a": 1, "b": "x"}), (2, {"a": 2, "b": "y"})])["objects"])
    anchor = db.record_anchor(11, 2)
    print(anchor["u"], anchor["v_local"])            # record 2 starts at v = (2-1)*8 + 1
    print(db.delete_record(11, 1), db.get_object(11, 1), db.get_object(11, 2)["b"])
    db.commit()

{'_layout': 10, '_layout_name': 'metrics', '_kind': 'COLUMN', '_object': 42, '_lane': 0, '_lane_name': 'NL', 'temp': 21.5, 'humidity': 40}
1
2
1 9
2 None y


*Erwartete Ausgabe (aus dem Cookbook):*

```text
{'_layout': 10, '_layout_name': 'metrics', '_kind': 'COLUMN', '_object': 42, '_lane': 0, '_lane_name': 'NL', 'temp': 21.5, 'humidity': 40}
1
2
1 9
2 None y
```

**Mehrere Layouts in einem Realm (Routing).** Layouts können sich einen Realm mit
unterschiedlichen `u`-Bereichen teilen. Für ein gegebenes `u` gewinnt das Layout mit der höheren
`priority`, danach der engere Bereich, danach die höhere ID.

In [5]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "routing.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "mixed")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u32"}])
    db.create_classic_layout(1, 1, "all", shape_id=1)                          # u = 1 … ∞
    db.create_layout(2, 1, "special", "CLASSIC", shape_id=1, u_start=100, u_end=199, priority=5)

    print(db.layout_at(1, 50).name, db.layout_at(1, 150).name)
    print([l.name for l in db.layout_candidates(1, 150)])
    print(db.route_address(1, 150, 1)["layout_name"])
    print(db.layout_diagnostics(1)["summary"])
    print(db.get_layout(layout_id=2)["u_range"])
    lay = db.get_layout(layout_id=2)
    print(db.layout_at(1, 150).to_dict()["priority"], db.layout_at(1, 150).contains_u(150))

all special
['special', 'all']
special
{'realms': 1, 'overlaps': 1, 'conflicts': 0, 'gaps': 0}
{'from': 100, 'to': 199}
5 True


*Erwartete Ausgabe (aus dem Cookbook):*

```text
all special
['special', 'all']
special
{'realms': 1, 'overlaps': 1, 'conflicts': 0, 'gaps': 0}
{'from': 100, 'to': 199}
5 True
```

**Ein Layout sicher ändern.** `plan_layout_evolution` zeigt die Wirkung einer
Änderung, ohne sie anzuwenden; `evolve_layout` wendet sie an. Änderungen, die bereits
gespeicherten Daten einen anderen Layout-Eigentümer geben würden, werden mit
`SchemaError` abgelehnt.

In [6]:
import os
import tempfile

import nhydb

path = os.path.join(tempfile.mkdtemp(), "evolve.nhydb")
with nhydb.create(path) as db:
    db.create_realm(1, "r")
    db.create_shape(1, 1, "s", [{"v": 1, "name": "x", "type": "u32"}])
    db.create_layout(1, 1, "first", "CLASSIC", shape_id=1, u_start=1, u_end=1000)
    db.put(1, 5, [7], commit=True)

    plan = db.plan_layout_evolution(1, u_end=None)
    print(plan["status"], plan["routing_changes"][0]["from"])
    print(db.evolve_layout(1, u_end=None, description="open ended", commit=True))
    try:
        db.evolve_layout(1, u_start=10)                  # would orphan the record at u=5
    except nhydb.SchemaError:
        print("rejected")

safe 1001
LayoutDef(layout_id=1, realm_id=1, name="first", kind='CLASSIC', u_start=1, u_end=None, priority=0)
rejected


*Erwartete Ausgabe (aus dem Cookbook):*

```text
safe 1001
LayoutDef(layout_id=1, realm_id=1, name="first", kind='CLASSIC', u_start=1, u_end=None, priority=0)
rejected
```

`evolve_layout(..., add_fields=[...])` fügt einem COLUMN-/PACKED-/CUSTOM-Layout
Felder hinzu; `spacemap_update` gilt nur für CUSTOM-Layouts.

**CUSTOM-Layouts** übergeben die Abbildung an ein Python-Objekt, das du
zur Laufzeit registrierst (gespeichert wird nur der Plugin-Name). Das Plugin braucht
`logical_address(layout, object_id, field, lane)` und zum Lesen
`type_of(layout, address)` und `describe(layout, address)`. Optional:
`spacemap` und `occupied_addresses(db, layout, object_id, lane)`.

In [7]:
import os
import tempfile

import nhydb


class KeyValuePairs:
    """Object k stores 'key' at v = 2k-1 and 'val' at v = 2k, all in u = 100."""

    def logical_address(self, layout, object_id, field, lane):
        offset = {None: 0, "key": 0, "val": 1}[field]
        return {"u": 100, "v": (object_id - 1) * 2 + offset + 1, "lane": lane}

    def type_of(self, layout, address):
        return "utf8"

    def describe(self, layout, address):
        v = address["v_local"]
        return {"record": (v - 1) // 2 + 1, "field": ("key", "val")[(v - 1) % 2], "anchor": (v - 1) % 2 == 0}


path = os.path.join(tempfile.mkdtemp(), "custom.nhydb")
with nhydb.create(path) as db:
    db.create_realm(50, "kv")
    db.create_custom_layout(9, 50, "pairs", plugin="kv-pairs", u_start=100, u_end=100,
                            params={"fields": [{"name": "key"}, {"name": "val"}]})
    db.register_layout_plugin("kv-pairs", KeyValuePairs())   # needed again after every open
    db.put_object(9, 3, {"key": "color", "val": "blue"}, commit=True)
    print(db.get_object(9, 3)["val"], db.logical_address(9, 3, "val")["v_local"])
    print(db.get_n(db.logical_address(9, 3, "key")["N"])["data"])

blue 6
{'key': 'color', 'val': 'blue'}


*Erwartete Ausgabe (aus dem Cookbook):*

```text
blue 6
{'key': 'color', 'val': 'blue'}
```

Ohne registriertes Plugin lösen Operationen, die die Abbildung brauchen,
`SchemaError("CUSTOM layout requires registered plugin '…'")` aus. Ein Plugin
darf aus einem Callback heraus nicht in dieselbe Datenbank schreiben.

`create_layout(...)` ist die allgemeine Form aller `create_*_layout`-Methoden
(`kind="CLASSIC" | "COLUMN" | "PACKED" | "CUSTOM"`, `replace=True`, um
ein Layout neu zu definieren, dessen Adressraum noch leer ist).